# 3 - A partir das extremidades das águas correntes e das águas correntes estimadas, traçar um círculo de 50m;

o find_nascentes quase que deu certo, mas tem um erros e antes de eu resolver esses erros eu preciso encontrar onde eles estõ acontecendo... bom, vamos começar usando um específico como exemplo, o do CORREGO MANDAQUI

In [1]:
import geopandas as gpd
import pandas as pd
import shapely
import os
from tqdm import tqdm


In [2]:
def testar_gdf(gdf):
    print(
    f'Shape: {gdf.shape};\n'+
    f'\nSample:{gdf.sample()}'
)

# Raw gdf

In [3]:
drenageo = gpd.read_file(
    os.path.join(
        'data',
        'drenagem.zip'
    )
)

# Silver gdf

In [4]:
## Como as correntes estimadas tem a id como int, vou transformar a id do drenageo em int tbm
drenageo['cd_identif'].astype('int', copy=False)
## Conferir se todos os 'cd_tipo_cu' sejam do mesmo tipo
drenageo['cd_tipo_cu'].dtype

dtype('float64')

# Determinar Correntes Estimadas (só por enquanto, dps vamos usar o do Elias e tals) 

In [5]:
drenageo.sample(10)
#* 11: trecho em estado natural
#* 12: lago ou reservatório
#* 10: trecho fechado
#* 9: trecho a céu aberto

cus_to_keep = [9.0, 11.0]
colors_dictionarie= {
    9.0 : 'turquoise',
    11.0 : 'aquamarine',
    10.0 : 'pink',
    12.0 : 'pink',
}

drenageo['colors'] = drenageo['cd_tipo_cu'].map(colors_dictionarie)

# Create GDF copy

In [6]:
gdf= drenageo[[
    'cd_identif', 
    'cd_tipo_ac', 
    'cd_tipo_cu', 
    'nm_acident', 
    'geometry'
]]

# Erro 1: Pontos buff

In [7]:
erro_pontos = gdf.copy()
erro_pontos_buff = gdf.copy()
erro_pontos['geometry']=shapely.get_point(gdf.geometry, -1)

erro_pontos_buff['geometry'] = erro_pontos['geometry'].buffer(10)

Que engraçado. Encontramos o erro e ele tá aqui em cima, não lá embaixo, onde eu esperava.

Minha teoria: o shapely só pega uma das extremidades por ponto, não duas, como eu acho que deveria ser o certo. 

Descobrimos!!! Aaaah, o ponto é que se passar o `get_point()` com `0`, aí, vai retornar o primeiro ponto da linha e com o `-1` vai retornar o último ponto da linha... por isso que não estava dando certo.

# V.3. Agora nosso problema está sendo (representado por) essa VILA BARBOSA:

Após conferir isso lá embaixo, realmente o VILA BARBOSA não tem nada nela, então precisamos descobrir o motivo.

In [8]:
teste=drenageo.loc[drenageo['cd_identif']==1649]
ponteste = shapely.get_point(teste.geometry, 0)
ponteste_fim = shapely.get_point(teste.geometry, -1)

In [9]:
gdf_ponto = gpd.GeoDataFrame(geometry=ponteste)
gdf_ponto.set_crs(teste.crs, inplace=True)  # usa o CRS do original, se definido

gdf_ponfim = gpd.GeoDataFrame(geometry=ponteste_fim)
gdf_ponfim.set_crs(teste.crs, inplace=True)

,geometry
1771,POINT (329606.913 7400740.117)


## visualizar teste

m= teste.explore(color='orange')
gdf_ponto.explore(
    m=m
)

gdf_ponfim.explore(
    m=m,
    color='purple'
)

In [10]:
pontos = gdf.copy()
pontos_0 = gdf.copy()
pontos_1 = gdf.copy()

pontos_0['geometry']=shapely.get_point(gdf.geometry, 0)
pontos_0['cd_point'] = pontos_0['cd_identif'].astype(int).astype(str)+".0"
pontos_1['geometry'] = shapely.get_point(gdf.geometry, -1)
pontos_1['cd_point'] = pontos_1['cd_identif'].astype(int).astype(str)+".1"
pontos = pd.concat([pontos_0, pontos_1], ignore_index=True)

pontos_buff = pontos.copy()
pontos_buff['geometry'] = pontos['geometry'].buffer(10)

## visualizar 2.0

#### já deu erro no pontos_buff.explore()
#### mas o pontos.explore() tá certo!! (foi burrice minha)
#### Vamos tentar agr com 1
m= pontos.explore()
teste.explore(m=m, color='red')
### oxi, o ponto 1 já ta mostrando os dois... deixa eu ver se isso acontece com o ponto 0
##### ENTÃO: o que eu to vendo ali que eu to achando que são 2 pontos, na vdd é o ponto final da linha + ponto final da linha vizinha
### NO 0 só aparece um... 


## (DONE) Tarefa:
comparar os shapes/ len() em: 
* pontos
* ponto_0
* ponto_1
Precisamos levar em consideração o shape das linhas... deveria ter o dobro, certo?

In [11]:
print(
    f'Shape das linhas: {gdf.shape}\n'
    + f'O dobro disso é: {gdf.shape[0]*2}'
)
print(
    f'Shape dos pontos_0: {pontos_0.shape}\n'
     + 'Conclusão: pontos_0 só retorna um ponto por linha'
)
print(
    f'Shape dos pontos_1: {pontos_1.shape}\n'
    + 'Conclusão: pontos_1 também só retorna um ponto por linha TAMBÉM'
)
print(
    f'Shape dos pontos: {pontos.shape}\n'
    + f'Shape dos pontos_buff: {pontos_buff.shape}\n'
    + f'Eles são iguais? {pontos.shape[0]==pontos_buff.shape[0]}\n'
    + 'Conclusão: PARECE que em pontos aparece tudo... vamos voltar na visualização pra ver se isso tá certo'
    + '\n(spoiler: está, eu que fui burra)'
    
)



Shape das linhas: (27611, 5)
O dobro disso é: 55222
Shape dos pontos_0: (27611, 6)
Conclusão: pontos_0 só retorna um ponto por linha
Shape dos pontos_1: (27611, 6)
Conclusão: pontos_1 também só retorna um ponto por linha TAMBÉM
Shape dos pontos: (55222, 6)
Shape dos pontos_buff: (55222, 6)
Eles são iguais? True
Conclusão: PARECE que em pontos aparece tudo... vamos voltar na visualização pra ver se isso tá certo
(spoiler: está, eu que fui burra)


Ok, acho que agora tá tudo corrigido
## Qual foi a burrada?
Ao invés de fazer o pontos_buff a partir dos pontos, eu estava fazendo a partir do drenageo, então eles só pegavam alguns dos pontos, mudando a geometria das linhas pros primeiros pontos respectivos. Ai, burrada.
## Correção:
`pontos_buff = pontos.copy()` 

# v.4. Agora o erro está em trechos que não foram pegos
Minha teoria:
1. O buffer é muito grande. Então só precisamos ir testando buffers menores e ir vendo se o quando o shape aumenta e se aumenta mt e se não perde nada

In [12]:
#E vamos, claro, conferir as intersecções
intersecs_bool=[]

for i, row in pontos_buff.iterrows():
    outras_geoms = pontos_buff.loc[pontos_buff.index!=i]
    outras_geoms.sample()
    intersecs_bool = row.geometry.intersects(outras_geoms.geometry)
    if len(intersecs_bool.loc[intersecs_bool==True])<1:
        pontos_buff.loc[pontos_buff.index==i, 'intersec_bool'] = (
            False
        )
    else:
        pontos_buff.loc[pontos_buff.index==i, 'intersec_bool'] = (
            True
        )

In [13]:
pontos_buff.sample()

,cd_identif,cd_tipo_ac,cd_tipo_cu,nm_acident,geometry,cd_point,intersec_bool
54822,26000.0,ND,11.0,SD,"POLYGON ((332645.281 7405853.554, 332645.233 7...",26000.1,True


In [14]:
for i, row in pontos_buff.loc[pontos_buff['intersec_bool']==False].iterrows():
    outras_linhas = gdf.loc[gdf['cd_identif']!=row['cd_identif']]
    outras_linhas
    intersecs_bool = row.geometry.intersects(outras_linhas.geometry)
    if len(intersecs_bool.loc[intersecs_bool==True])<1:
        pontos_buff.loc[pontos_buff.index==i, 'intersec_bool']= False
    else:
        pontos_buff.loc[pontos_buff.index==i, 'intersec_bool'] = True

In [15]:
pontos_buff.loc[pontos_buff['intersec_bool']!=False]

,cd_identif,cd_tipo_ac,cd_tipo_cu,nm_acident,geometry,cd_point,intersec_bool
1,26125.0,ND,11.0,SD,"POLYGON ((332074.747 7351146.304, 332074.699 7...",26125.0,True
2,2.0,ND,11.0,SD,"POLYGON ((345331.612 7399298.312, 345331.564 7...",2.0,True
3,26126.0,COR,10.0,CORREGO PIRARUNGAUA,"POLYGON ((334522.87 7384823.742, 334522.822 73...",26126.0,True
5,26152.0,ND,11.0,SD,"POLYGON ((320684.197 7386065.237, 320684.149 7...",26152.0,True
6,26153.0,ND,10.0,SD,"POLYGON ((327674.456 7379012.447, 327674.408 7...",26153.0,True
...,...,...,...,...,...,...,...
55217,27596.0,ND,11.0,SD,"POLYGON ((331460.637 7349720.122, 331460.589 7...",27596.1,True
55218,27597.0,ND,11.0,SD,"POLYGON ((322431.727 7346178.926, 322431.679 7...",27597.1,True
55219,27598.0,ND,11.0,SD,"POLYGON ((317873.595 7373040.717, 317873.547 7...",27598.1,True
55220,27599.0,ND,11.0,SD,"POLYGON ((333150.363 7375883.084, 333150.315 7...",27599.1,True


In [16]:
pontos_buff.shape

(55222, 7)

In [17]:
pontos_buff['cd_tipo_cu'].astype(dtype='float', copy=False)
pontos_buff= pontos_buff.loc[pontos_buff['cd_tipo_cu'].isin(cus_to_keep)]
pontos_buff = pontos_buff.loc[pontos_buff['intersec_bool']==False]

In [18]:
pontos_buff.shape

(8382, 7)

# Visualizar
### Ok, eu estava errada... mesmo com o intersec e um mega buffer nos pontos, ainda não dá certo, vamos voltar pra tatica do Henrique mesmo
m= drenageo.explore(color='pink')

drenageo.loc[drenageo['cd_tipo_cu'].isin(cus_to_keep)].explore(
    m=m,
    color='purple'
)

drenageo.loc[drenageo['nm_acident']=="CORREGO MANDAQUI"].explore(m=m, color='orange')

teste.explore(m=m, color='red')

pontos_buff.explore(
    m=m,
    color="green"
)

E depois disso tudo, aquele bendito riozinho que o Mauryas achou ainda não foi pego! Vou fazer um commit pro que eu já fiz e depois passo as coreeções daqui lá pro find_nascentes, e depois procurar os motivos desse novo erro

In [19]:
# Salvar arquivo para a validação do Mauryas
pontos_buff.to_file(
    os.path.join(
        'data',
        'pontos_buff_v4.0.geojson'
    ),
    driver="GeoJSON"
)